In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config/config.yaml").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.runtime import configure_runtime
configure_runtime(headless=False)
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass


# Study Result Inspection
Use `python -m src.study --config config/full.yaml` to generate a complete study. Smoke manifests are execution diagnostics only. Inspect one manifest at a time; never pool unrelated result files.


In [ ]:
import json
import pandas as pd
from src.report import assemble_report
manifest_files = sorted((PROJECT_ROOT / "outputs/results").glob("*_manifest.json"))
for path in manifest_files:
    info = json.loads(path.read_text(encoding="utf-8"))
    print(path.name, info["status"], "smoke=" + str(info["smoke"]))


In [ ]:
# Set this explicitly to the manifest printed by the study runner.
manifest_path = None
if manifest_path is None:
    raise ValueError("Set manifest_path to a complete study manifest before comparing results.")
manifest_path = Path(manifest_path)
if not manifest_path.is_absolute():
    manifest_path = PROJECT_ROOT / manifest_path
report_dir = assemble_report(manifest_path)
report = json.loads((report_dir / "report.json").read_text(encoding="utf-8"))
print(report["conclusion"])
print("Seeds:", report["seeds"], "Smoke:", report["smoke"])


In [ ]:
pd.read_csv(report_dir / "main_table.csv")


In [ ]:
pd.read_csv(report_dir / "ablation_table.csv")


In [ ]:
scenarios = pd.read_csv(report_dir / "scenario_table.csv")
scenarios


Negative PI-minus-baseline differences favor physics. Primary tables use raw predictions. Check run IDs, seed dispersion, paired intervals, and underpowered strata before interpretation. nRMSE differences are percentage points. Read `docs/research.md` for limitations.
